## Ejercicio 1 a)

### Planteo del sistema

Para cada punto $\tilde{\mathbf P}_i=(U_i,V_i,W_i,1)^T$ con píxel $(u_i,v_i)$, el modelo pinhole da $\lambda\tilde{\mathbf m}_i=M\tilde{\mathbf P}_i$, con $M\in\mathbb R^{3\times4}$ y filas $\mathbf m_1^T,\mathbf m_2^T,\mathbf m_3^T$. Eliminando $\lambda$:

$$u_i=\frac{\mathbf m_1^T\tilde{\mathbf P}_i}{\mathbf m_3^T\tilde{\mathbf P}_i},\qquad v_i=\frac{\mathbf m_2^T\tilde{\mathbf P}_i}{\mathbf m_3^T\tilde{\mathbf P}_i}
\;\Longrightarrow\;
(\mathbf m_1-u_i\mathbf m_3)^T\tilde{\mathbf P}_i=0,\quad(\mathbf m_2-v_i\mathbf m_3)^T\tilde{\mathbf P}_i=0$$

Estas dos ecuaciones son lineales en las incógnitas. Con $\mathbf m=(\mathbf m_1;\mathbf m_2;\mathbf m_3)\in\mathbb R^{12}$, cada punto aporta dos filas:

$$\begin{bmatrix}\tilde{\mathbf P}_i^T&\mathbf 0^T&-u_i\tilde{\mathbf P}_i^T\\\mathbf 0^T&\tilde{\mathbf P}_i^T&-v_i\tilde{\mathbf P}_i^T\end{bmatrix}\mathbf m=\begin{bmatrix}0\\0\end{bmatrix}$$

Apilando los $n$ puntos se obtiene $Q\mathbf m=0$ con $Q\in\mathbb R^{2n\times12}$. Con los datos de `calib/3d` (12 puntos por imagen), $Q$ resulta de tamaño $24\times12$.

### a) ¿Qué rango debe tener $Q$?

$M$ y $\alpha M$ ($\alpha\neq0$) producen las mismas proyecciones, porque $\alpha$ se cancela al dividir por la tercera componente. Por lo tanto $M$ solo puede determinarse a menos de un factor de escala, y queremos que la solución $\mathbf m\neq0$ sea única salvo escala. Esto equivale a que el núcleo de $Q$ tenga dimensión 1:

$$\dim\ker Q=1.$$

Por el teorema de la dimensión, con $12$ columnas:

$$\operatorname{rango}(Q)+\dim\ker Q=12\;\Longrightarrow\;\operatorname{rango}(Q)=12-1=11.$$

(Coincide con los grados de libertad de $M$: $12$ entradas menos $1$ de escala. También con $5$ de $K$, $3$ de $R$ y $3$ de $\mathbf t$.)

### ¿Cuántos puntos como mínimo?

Se necesitan $11$ ecuaciones linealmente independientes. Cada punto aporta $2$ ecuaciones:

$$2n\ge11\;\Longrightarrow\;n\ge5{,}5\;\Longrightarrow\;n\ge6.$$

Con $n=5$ hay solo $10$ ecuaciones y el rango es a lo sumo $10<11$, entonces el núcleo tendría dimensión $\ge2$ y la solución no sería única salvo escala. Con $n=6$ hay $12$ filas y puede alcanzarse rango $11$.

### ¿Qué condición deben cumplir los puntos?

El conteo es necesario pero no suficiente: las ecuaciones deben ser linealmente independientes. Para eso los puntos tienen que estar en **posición general** (no están en una configuración geométrica especial que impida determinar la cámara), y en particular:

- **No pueden ser coplanares.** Si todos están en un plano, por ejemplo $W=0$, las tres columnas de $Q$ que multiplican a $W$ son nulas y el rango queda en a lo sumo $9<11$. Por eso el caso coplanar se trata aparte (Ejercicio 3, mediante una homografía).
- No deben estar, junto con el centro de la cámara, sobre una cúbica alabeada, que es la otra configuración crítica clásica del DLT.
- No deben repetirse puntos, porque una fila repetida no aporta una ecuación nueva.

En nuestros datos, los $12$ puntos se reparten en tres planos distintos ($W=0$, $V=26{,}6$ y $U=19{,}4$), por lo que no son coplanares y se cumple la condición.

### Resumen

| Pregunta | Respuesta |
|---|---|
| Rango de $Q$ | $11$ ($\dim\ker Q=1$) |
| Mínimo de puntos | $n\ge6$ |
| Condición | Posición general, no coplanares |



## Ejercicio 1 b)

Buscamos el vector $\mathbf m$ que resuelve la minimización de la ecuación 8:

$$
\min_{\|\mathbf m\|=1}\ \mathbf m^T Q^T Q\,\mathbf m
$$

**Por qué no alcanza con derivar.** Sin la restricción, derivar $\mathbf m^TQ^TQ\mathbf m$ e igualar a $0$ lleva a $Q^TQ\,\mathbf m=0$. Con datos reales (con ruido) $Q^TQ$ es invertible, así que la única solución es $\mathbf m=\mathbf 0$. Esa solución es inútil (no es una cámara), y la restricción $\|\mathbf m\|=1$ sirve justamente para descartarla.

### Uso de la SVD

Sea $Q=U\Sigma V^T$ la descomposición en valores singulares de $Q$, con $U$ y $V$ ortogonales. Entonces, como $U^TU=I$:

$$
Q^TQ=(V\Sigma^TU^T)(U\Sigma V^T)=V\,\Sigma^T\Sigma\,V^T
$$

### Cambio de coordenadas

Las columnas de $V$ son ortonormales y forman una base de $\mathbb R^{12}$, así que cualquier $\mathbf m$ puede escribirse como combinación lineal de ellas:

$$
\mathbf m=V\mathbf c
$$

Multiplicando a ambos lados por $V^T$ y usando $V^TV=I$, obtenemos

$$
\mathbf c=V^T\mathbf m
$$

Reemplazando en la función a minimizar:

$$
\mathbf m^TQ^TQ\,\mathbf m=\mathbf c^T\Sigma^T\Sigma\,\mathbf c=\sum_{k}\sigma_k^2\,c_k^2
$$

### La restricción en las nuevas coordenadas

Como $V$ es ortogonal, preserva la norma:

$$
\|\mathbf c\|=\|V^T\mathbf m\|=\|\mathbf m\|=1
$$

Elevando al cuadrado, esto equivale a

$$
\sum_k c_k^2=1
$$

Entonces el problema se reduce a:

$$
\min\ \sum_k\sigma_k^2\,c_k^2\quad\text{sujeto a}\quad\sum_k c_k^2=1
$$

### Resolución

Los $c_k^2\ge0$ suman $1$, así que $\sum_k\sigma_k^2c_k^2$ es un promedio ponderado de los $\sigma_k^2$. Un promedio nunca es menor que el menor de sus valores, y ese valor se alcanza poniendo todo el peso en él. Con los valores singulares ordenados de mayor a menor, el menor es $\sigma_{12}$, por lo que:

$$
\sum_k\sigma_k^2c_k^2\ \ge\ \sigma_{12}^2\sum_kc_k^2=\sigma_{12}^2
$$

con igualdad cuando

$$
\mathbf c=(0,0,\dots,0,\pm1)
$$

Volviendo a la variable original:

$$
\mathbf m=V\mathbf c=\pm\,\mathbf v_{12}
$$

**Conclusión:** $\mathbf m$ es la última columna de $V$, es decir, el vector singular derecho de $Q$ asociado al menor valor singular (equivalentemente, el autovector de $Q^TQ$ asociado a su menor autovalor). El error mínimo vale $\sigma_{12}^2$. El signo queda indeterminado, ya que $M$ y $-M$ producen la misma proyección.

### 2 a) obtenerQ3d

In [ ]:
import numpy as np
from alc import obtenerQ3d
# Puntos 3D del mundo: (12, 3)
objPts = np.loadtxt("TP_template/ calib/3d/p_W_corners.txt", delimiter=",")

# Píxeles detectados: (210, 24), una fila por imagen
corners = np.loadtxt("TP_template/ calib/3d/detected_corners.txt")

# Elegimos una imagen (fila 0) y la pasamos a 12 pares (u, v): (12, 2)
imgPts = corners[0].reshape(12, 2)

print(objPts.shape, imgPts.shape)   # (12, 3) (12, 2)

Q = obtenerQ3d(imgPts, objPts)
print(Q.shape)                      # (24, 12)

(12, 3) (12, 2)
(24, 12)


### 2 b) resolverDLT3D(Q)

No hace falta calcular la SVD completa: alcanza con los autovectores de $Q^TQ$, que son las columnas de $V$. Usamos `diagRH`, que los devuelve ordenados de mayor a menor autovalor, y nos quedamos con la última columna (la del menor autovalor). Ese vector es $\mathbf m$, y como apila las filas de $M$, lo reacomodamos como una matriz de $3\times4$.

In [ ]:
from alc import resolverDLT3D

M = resolverDLT3D(Q)
print(M.shape)   # (3, 4)
print(M)

### 2 c) devolverMatricesCalibracion3d(M)

Mirando la ec. 4, $K$ es triangular superior y $R$ es ortogonal (matriz de rotación: columnas de norma 1 y ortogonales entre sí).

Como $M = K\,[R \mid \mathbf t] = [\,KR \mid K\mathbf t\,]$, separamos:

- $A = M_{[:,\,0:3]} = KR$
- $\mathbf b = M_{[:,\,3]} = K\mathbf t$

**Obtener $K$ y $R$.** Invirtiendo $A$:

$$A^{-1} = R^{-1}K^{-1} = R^T K^{-1}$$

- $R^T$ es ortogonal, porque $R^T = R^{-1}$ y la inversa de una ortogonal es ortogonal.
- $K^{-1}$ es triangular superior, porque la inversa de una triangular superior también lo es.

Entonces $A^{-1}$ = (ortogonal) $\times$ (triangular superior), que es exactamente una factorización QR. Si $A^{-1} = Q_{qr}R_{qr}$:

$$R = Q_{qr}^T, \qquad K = R_{qr}^{-1}$$

**Obtener $\mathbf t$.** De $\mathbf b = K\mathbf t$ despejamos $\mathbf t = K^{-1}\mathbf b = R_{qr}\,\mathbf b$. No hace falta otra inversa, porque $K^{-1}$ es la $R_{qr}$ que ya tenemos.

La QR la calculamos con reflectores de Householder (`QR_con_HH`) y la inversa con la descomposición LU (`inversa`).

Ojo, acá hay que tener en cuenta algo. Cuando obtuvimos m teníamos la restricción de que su norma sea 1, y por eso diagRH nos devuelve autovectores de norma 1.

Pero la M verdadera, la que tiene K[2,2] = 1, no tiene por qué tener norma 1. Como Qm = 0 es homogéneo, si m lo cumple también lo cumplen 2m, -m, 0.001m, etc. Todos apuntan para el mismo lado, pero tienen distinta norma (multiplicar por un escalar sí cambia la norma: ||a·m|| = |a|·||m||). De todos esos múltiplos, el método nos devuelve justo el que tiene norma 1:

m_encontrado = ± m_verdadero / ||m_verdadero||

O sea que nuestra M en realidad es c·K[R | t], con c = ±1/||m_verdadero||, y ese c se nos mete en K. Se nota porque K[2,2] no nos da 1 como dice la ec. 4, nos da justamente c. Así que para sacarlo dividimos K por K[2,2] y listo.

A R y a t no hace falta tocarlos. R sale de la QR, que siempre da una matriz ortonormal, así que no tiene escala. Y en t el c se cancela solo, porque la última columna de M también viene multiplicada por c y la multiplicamos por K⁻¹, que trae 1/c.

Falta una cosa más: el signo. El método nos puede devolver m o -m, porque los dos tienen norma 1 y los dos cumplen Qm = 0. Cuál nos toca es medio azar. Si nos toca -m, el c es negativo.

Al dividir K por K[2,2] el c se va de K, pero el signo menos no desaparece: se pasa a R y a t, y nos quedan -R y -t en vez de R y t. Esto está mal por dos lados:

- -R no es una rotación. det(-R) = (-1)³·det(R) = -1, y una matriz ortogonal con determinante -1 es una reflexión (como un espejo). La cámara no puede espejar el mundo.
- t₃ < 0 querría decir que el objeto está detrás de la cámara, lo cual es imposible porque lo estamos viendo en la foto.

El arreglo es fácil: si det(R) < 0, damos vuelta R y t. No rompemos nada, porque K[R | t] y K[-R | -t] proyectan igual: es multiplicar todo por -1, y eso se cancela al dividir por λ. Como no podemos usar np.linalg.det, el determinante de 3x3 lo calculamos a mano (`determinante_3x3`, desarrollando por la primera fila).


In [ ]:
from alc import devolverMatricesCalibracion3d

np.set_printoptions(precision=3, suppress=True)

R, t, K = devolverMatricesCalibracion3d(M)
print("K =\n", K)
print("R =\n", R)
print("t =", t)

### 2 d) Agregar a la función devolverMatricesCalibracion3d un parámetro opcional que permita imponer que αu = αv y/o K12 = 0.

Es un arreglo sencillo, Sacamos el promedio entre ambas alphas.

Los alphas deben ser iguales porque estan midiendo ambos la distancia focal, uno con la distancia del centro con respecto a los pixeles horizontales y otro con los verticales.
Ocurre estos errores por ruidos, errores en las esquinas de la foto.

## Ejercicio 3

En el ejercicio 2 teníamos una mira con puntos en 3D. Acá el objeto es un tablero de ajedrez plano, así que todos los puntos tienen W = 0. Como vimos en el 1a, con puntos coplanares el DLT del ejercicio 2 no anda: las columnas de Q que multiplican a W quedan en cero y la tercera columna de M no se puede determinar.

La salida es mirar la ec. 9. Si W = 0, la tercera columna de R (r₃) siempre se multiplica por 0, así que es como si no estuviera. Nos queda H = K[r₁ r₂ t], que es de 3x3: una homografía entre el plano del tablero y la foto.

### ¿Qué rango debe tener Q ∈ R^{2n×9} para asegurar una solución única de H a menos de un factor de escala?

Queremos que todas las soluciones de Qh = 0 sean múltiplos de un mismo vector, o sea que el núcleo de Q sea una recta: dimensión 1. Por el teorema de la dimensión, rango(Q) + dim Nu(Q) = 9, así que Q tiene que tener rango 8.

Es lo mismo que contar incógnitas: H tiene 9 entradas pero una se pierde por la escala, quedan 8. Cada punto da 2 ecuaciones, así que hacen falta al menos 4 puntos, y que no haya 3 alineados (si no, las ecuaciones dejan de ser independientes). Con el tablero tenemos 77 esquinas por foto, así que sobra.

### obtenerQ2d(imgPts, objPts)

imgPts dice dónde aparece cada esquina en la foto (en píxeles, (u, v)) y objPts dónde está esa misma esquina en el tablero real (en mm, (U, V), con W = 0 así que no hace falta guardarlo). La fila k de los dos tiene que ser la misma esquina, si no H sale cualquier cosa.

Es igual que obtenerQ3d pero con P = (U, V, 1): cada punto aporta dos filas [P, 0, -uP] y [0, P, -vP], y Q queda de 2n x 9.

### armarA(Hlst)

Con una sola foto no alcanza para sacar K. Cada H nos da 8 números, pero en cada foto hay 5 incógnitas de K y 6 de la pose (3 de la rotación y 3 de la traslación): 11 incógnitas para 8 ecuaciones. La clave es que K es la misma en todas las fotos (es la misma cámara) y la pose no. Con N fotos tenemos 8N ecuaciones y 5 + 6N incógnitas, así que necesitamos 8N ≥ 5 + 6N, o sea N ≥ 3 fotos.

¿De dónde salen las ecuaciones para K? De que r₁ y r₂ son columnas de una rotación: son perpendiculares y tienen el mismo largo. Como h₁ = K r₁ y h₂ = K r₂, despejamos r₁ = K⁻¹h₁ y r₂ = K⁻¹h₂, y lo metemos en esas dos condiciones:

- perpendiculares: h₁ᵀ B h₂ = 0
- mismo largo: h₁ᵀ B h₁ = h₂ᵀ B h₂

con B = K⁻ᵀK⁻¹. La pose desapareció y solo queda K, metida adentro de B.

Usamos "mismo largo" y no "largo 1" porque H la conocemos a menos de una escala (igual que M en el ejercicio 2), así que no sabemos cuánto miden h₁ y h₂ en serio. Pero las dos columnas vienen multiplicadas por la misma escala, así que comparar sus largos sí se puede.

Trabajamos con B en vez de K porque estas ecuaciones son lineales en B (en K serían cuadráticas). B es simétrica de 3x3, así que tiene 6 valores distintos: b = (B₁₁, B₁₂, B₂₂, B₁₃, B₂₃, B₃₃). Cada condición se puede escribir como un producto escalar v · b, y eso es lo que arma construirVij. Cada foto aporta 2 filas (v₁₂ y v₁₁ - v₂₂), y apilando todas queda A b = 0, con A de 2N x 6.

### calcularMatrizK2D(A)

A b = 0 es el mismo tipo de sistema que Qm = 0 del ejercicio 2, así que lo resolvemos igual: b es el autovector de AᵀA de menor autovalor, que sacamos con diagRH (la última columna).


Igual que en el ejercicio 2, b sale con norma 1, así que lo que tenemos es c·B y no B. El signo lo arreglamos mirando b[0] = B₁₁ = 1/αᵤ², que tiene que ser positivo: si sale negativo, damos vuelta b. Si no, c·B no sería definida positiva y Cholesky no andaría.

Después armamos B y usamos que B = K⁻ᵀK⁻¹. Cholesky nos da B = L Lᵀ con L triangular inferior, así que L = K⁻ᵀ y K = (Lᵀ)⁻¹. La escala c se cuela en K como 1/√c, y la sacamos dividiendo por K[2,2], que tiene que valer 1 (mismo truco que en el 2c).

### obtenerPose2D(Hj, K)

Ya con K, la pose de cada foto sale de H = λ K[r₁ r₂ t]: multiplicamos por K⁻¹ y nos queda λ[r₁ r₂ t]. Para encontrar la escala usamos que r₁ tiene que tener largo 1, así que λ = 1/‖K⁻¹h₁‖. Entonces:

- r₁ = λ K⁻¹h₁
- r₂ = λ K⁻¹h₂
- t = λ K⁻¹h₃
- r₃ no está en H (era la que se multiplicaba por W = 0), pero como R es una rotación, r₃ tiene que ser perpendicular a r₁ y a r₂: r₃ = r₁ × r₂ (producto vectorial).

r₁, r₂ y r₃ van como columnas de R. Y con el signo pasa lo mismo que en el 2c: H puede venir con el signo cambiado, y entonces t sale con t₃ < 0 (el tablero detrás de la cámara). En ese caso hay que elegir λ negativo, que da vuelta r₁, r₂ y t.


## Ejercicio 4

La idea es ubicar objetos sobre la cancha a partir de una foto.
Como la cancha es un plano, la relación entre sus puntos y los de la imagen queda determinada por una homografia H.




Para estimar H necesitamos al menos 4 puntos que sepamos dónde están en la cancha (en metros) y dónde aparecen en la foto (en píxeles). Usamos las líneas de cal, porque sus medidas son reglamentarias y están en `estadio_azteca.png`.

Tomamos el mismo sistema de coordenadas que ese esquema: el origen en la esquina de abajo a la izquierda, U a lo largo de la cancha (0 a 105 m) y V a lo ancho (0 a 68 m). En las dos fotos se ataca el arco de la derecha, así que la línea de meta es U = 105 y el arco está centrado en V = 34.

Las medidas que usamos:
- Área chica: 5,5 m de profundidad y 18,32 m de ancho, así que va de V = 24,84 a V = 43,16 y su línea de adelante está en U = 99,5.
- Área grande: 16,5 m de profundidad y 40,32 m de ancho, de V = 13,84 a V = 54,16, con la línea de adelante en U = 88,5.
- Arco: 7,32 m, los palos en V = 30,34 y V = 37,66.
- Medialuna: es un arco de radio 9,15 m alrededor del punto penal (U = 94, V = 34). Corta a la línea del área (que está 5,5 m más cerca de la mitad de cancha) en V = 34 ± √(9,15² − 5,5²) ≈ 34 ± 7,31.

Los píxeles los marcamos a mano haciendo zoom sobre cada foto. Con la H estimada volvemos a proyectar los puntos de la cancha y medimos cuánto se alejan de donde los marcamos (error de reproyección), para ver que H tenga sentido.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from alc import estimarHomografiaCancha, imagenACancha, proyectarPuntos, circuloEnCancha

img_gol = mpimg.imread("TP_template/futbol/test-gol-enzo.png")

medialuna = np.sqrt(9.15**2 - 5.5**2)   # media cuerda de la medialuna sobre la línea del área

# Puntos de la cancha (U, V) en metros
cancha_gol = np.array([[105,  43.16],               # área chica: esquina sobre la línea de meta (izq. en la foto)
                       [99.5, 43.16],               # área chica: esquina de adelante (izq.)
                       [105,  24.84],               # área chica: esquina sobre la línea de meta (der.)
                       [99.5, 24.84],               # área chica: esquina de adelante (der.)
                       [105,  37.66],               # base del palo izquierdo
                       [105,  30.34],               # base del palo derecho
                       [88.5, 34 + medialuna],      # medialuna cortando la línea del área (izq.)
                       [88.5, 34 - medialuna]])     # medialuna cortando la línea del área (der.)

# Los mismos puntos en la foto (u, v) en píxeles
img_pts_gol = np.array([[194, 171], [157, 231], [842, 190], [866, 250],
                        [389, 176.5], [654, 184], [147, 406], [851, 422]])

H_gol = estimarHomografiaCancha(img_pts_gol, cancha_gol)

# Error de reproyección: llevamos los puntos de la cancha a la foto con H
reproy = proyectarPuntos(H_gol, cancha_gol)
error = np.sqrt(np.sum((reproy - img_pts_gol)**2, axis=1))
print("Error de reproyección (px):", error)

plt.figure(figsize=(10, 6))
plt.imshow(img_gol)
plt.scatter(img_pts_gol[:, 0], img_pts_gol[:, 1], c="yellow", s=30, label="marcados a mano")
plt.scatter(reproy[:, 0], reproy[:, 1], c="red", marker="x", s=40, label="reproyectados con H")
plt.legend()
plt.title("Puntos de referencia - gol de Enzo")
plt.show()

### 4 a) Distancia de Enzo a la línea del arco

Marcamos en la foto el punto del piso donde está Enzo (el pie de apoyo, porque el otro está levantado pateando) y lo llevamos a la cancha con H⁻¹. Usamos los pies y no otra parte del cuerpo porque H solo vale para puntos que estén sobre el piso: la cabeza está a casi 2 metros de altura y H la mandaría a cualquier lado.

La línea del arco es U = 105, así que la distancia en línea recta (la perpendicular) es 105 − U. También calculamos la distancia hasta el centro del arco, (105, 34).

In [ ]:
pie_enzo = np.array([[806, 545]])   # pie de apoyo de Enzo (#24) en la foto

U_enzo, V_enzo = imagenACancha(H_gol, pie_enzo)[0]
print(f"Enzo está en la cancha en U = {U_enzo:.2f} m, V = {V_enzo:.2f} m")
print(f"Distancia a la línea del arco: {105 - U_enzo:.2f} m")
print(f"Distancia al centro del arco: {np.sqrt((105 - U_enzo)**2 + (34 - V_enzo)**2):.2f} m")

### 4 b) Círculos de 1 metro alrededor de los argentinos

Para cada jugador argentino (camiseta azul oscuro) marcamos dónde pisa en la foto y lo pasamos a la cancha con H⁻¹. Ese punto es el centro del círculo.

El círculo hay que armarlo en la cancha y no en la foto, porque en la foto un círculo del piso se ve como una elipse, y más chica cuanto más lejos esté el jugador. Entonces generamos los puntos del círculo de 1 m en metros (`circuloEnCancha`) y los llevamos a la foto con H (`proyectarPuntos`). Como H es una homografía, el círculo sale deformado justo como lo vería la cámara.

Uno de los argentinos (el de atrás en el grupo del medio) tiene los pies tapados por un compañero, así que su punto de apoyo lo estimamos por la altura del cuerpo.

In [ ]:
# Pies de los argentinos en la foto (u, v)
pies_argentinos = np.array([[103, 537],    # abajo a la izquierda
                            [242, 402],    # sobre la línea del área, a la izquierda
                            [378, 326],    # grupo del medio, adelante
                            [385, 285],    # grupo del medio, atrás (pies tapados, estimado)
                            [843, 352],    # a la derecha
                            [806, 545]])   # Enzo (#24)

centros = imagenACancha(H_gol, pies_argentinos)

plt.figure(figsize=(12, 7.5))
plt.imshow(img_gol)
for i in range(len(centros)):
    circulo = circuloEnCancha(centros[i], 1.0, 100)   # en metros, sobre el piso
    circulo_img = proyectarPuntos(H_gol, circulo)     # en píxeles
    plt.plot(circulo_img[:, 0], circulo_img[:, 1], c="cyan", lw=2)
    plt.text(pies_argentinos[i, 0] + 15, pies_argentinos[i, 1] + 20,
             f"({centros[i, 0]:.1f}, {centros[i, 1]:.1f})", c="white", fontsize=8)
plt.title("Argentinos con un círculo de 1 m de radio sobre el piso")
plt.show()

### 4 c) Posible fuera de juego

Esta foto es de otro partido, así que la H es otra. La armamos igual, con las esquinas del área grande y del área chica y el banderín del córner.

El que defiende es el equipo naranja (el arquero de celeste está en el arco de la derecha) y atacan los de blanco. Para el offside hay que mirar al penúltimo defensor contando al arquero. Como el arquero es el último, el penúltimo es el jugador naranja que esté más cerca de la línea de meta.

Para encontrarlo no alcanza con mirar la foto, porque por la perspectiva "más a la derecha en la foto" no es lo mismo que "más cerca del arco". Entonces pasamos los pies de todos los defensores a la cancha con H⁻¹ y nos quedamos con el de U más grande. Por esa U trazamos la recta paralela a la línea de fondo (U constante; la dibujamos de V = 3 a V = 68, porque más cerca de V = 0 se sale de la foto), y la llevamos a la foto con H. Como una homografía manda rectas en rectas, alcanza con proyectar los dos extremos.

Después hacemos lo mismo con los pies de los de blanco: si alguno tiene una U más grande que la del defensor, está más cerca del arco, o sea adelantado.

In [ ]:
img_off = mpimg.imread("TP_template/futbol/test-fuera-de-juego.png")

cancha_off = np.array([[88.5, 54.16],   # área grande: esquina de adelante (lejos)
                       [88.5, 13.84],   # área grande: esquina de adelante (cerca)
                       [105,  68],      # banderín del córner (lejos)
                       [105,  54.16],   # área grande: esquina sobre la línea de meta (lejos)
                       [99.5, 43.16],   # área chica: esquina de adelante (lejos)
                       [99.5, 24.84]])  # área chica: esquina de adelante (cerca)

img_pts_off = np.array([[736, 314], [735, 756], [1305, 236], [1385, 311], [1230, 389], [1365, 576]])

H_off = estimarHomografiaCancha(img_pts_off, cancha_off)
reproy = proyectarPuntos(H_off, cancha_off)
print("Error de reproyección (px):", np.sqrt(np.sum((reproy - img_pts_off)**2, axis=1)))

# Pies de los defensores de naranja (sin el arquero), el más cercano al arco de cada uno
pies_defensores = np.array([[823, 334], [762, 368], [712, 490], [766, 537], [815, 560]])
# Pies de los atacantes de blanco
pies_atacantes = np.array([[795, 350], [797, 470], [862, 510], [835, 535], [870, 590], [690, 410]])

defensores = imagenACancha(H_off, pies_defensores)
atacantes = imagenACancha(H_off, pies_atacantes)

# Penúltimo defensor: el de U más grande (el arquero es el último)
i_def = 0
for i in range(len(defensores)):
    if defensores[i, 0] > defensores[i_def, 0]:
        i_def = i
U_def = defensores[i_def, 0]
print(f"Penúltimo defensor: U = {U_def:.2f} m")

# Recta paralela a la línea de fondo que pasa por su pie.
# La dibujamos de V = 3 (más abajo se sale de la foto) a V = 68 (la línea lateral de enfrente)
recta_img = proyectarPuntos(H_off, np.array([[U_def, 3], [U_def, 68]]))

plt.figure(figsize=(12, 8.3))
plt.imshow(img_off)
plt.plot(recta_img[:, 0], recta_img[:, 1], c="red", lw=1.5, label="línea del penúltimo defensor")
plt.scatter(pies_defensores[:, 0], pies_defensores[:, 1], c="orange", s=25, label="defensores")
for i in range(len(atacantes)):
    adelantado = atacantes[i, 0] > U_def
    plt.scatter(pies_atacantes[i, 0], pies_atacantes[i, 1], c="red" if adelantado else "white",
                edgecolors="black", s=40)
    plt.text(pies_atacantes[i, 0] + 8, pies_atacantes[i, 1] + 18, f"{atacantes[i, 0] - U_def:+.2f} m",
             c="white", fontsize=8)
    print(f"Atacante en U = {atacantes[i, 0]:.2f} m -> {'ADELANTADO' if adelantado else 'habilitado'} "
          f"({atacantes[i, 0] - U_def:+.2f} m respecto del defensor)")
plt.legend()
plt.title("Línea de offside (en rojo los atacantes adelantados)")
plt.show()

**Conclusión.** Según nuestras cuentas, hay dos jugadores de blanco más cerca del arco que el penúltimo defensor, por unos 20 a 30 cm, así que estarían en fuera de juego. Igual hay que tomarlo con cuidado, porque es una diferencia chica y los pies los marcamos a mano. Para ver qué tan firme es el resultado probamos mover los pies: correrlos para arriba o para abajo en la foto casi no cambia U (la línea de offside se ve casi vertical, así que ese movimiento es casi paralelo a ella), y correrlos un píxel hacia los costados cambia U en unos 2 cm. Para que el de +0,19 m quedara habilitado habría que haberle errado por unos 10 píxeles, así que la conclusión se sostiene. Además, el reglamento mira cualquier parte del cuerpo con la que se pueda hacer un gol (no solo los pies) y el momento exacto del pase, y eso con una sola foto y solo los pies no lo podemos medir.

## Ejercicio 5

### 5 a) Parámetros intrínsecos con las vistas de la esquina

En `world_img_pts.csv` hay 5 puntos de la esquina. De cada uno sabemos dónde está en el mundo (X, Y, Z en metros) y dónde aparece en cada una de las 4 fotos (u_k, v_k). Lo que tenemos que sacar es K.

**¿Qué método usamos?** Primero hicimos la cuenta de ecuaciones contra incógnitas:

- **DLT 3D (ejercicio 2), foto por foto:** M tiene 11 incógnitas (12 menos la escala) y 5 puntos dan 10 ecuaciones. No alcanza. Y no se pueden juntar las fotos en un mismo sistema, porque cada una tiene su propia M (la pose cambia). Además los puntos están casi todos en el piso, y el DLT 3D pide puntos no coplanares.
- **Zhang (ejercicio 3):** tratando el piso como un plano, cada H tiene 8 incógnitas y 5 puntos dan 10 ecuaciones, así que alcanza. Para K, cada foto da 2 ecuaciones, o sea 8 con las 4 fotos, para 5 incógnitas.

Con los puntos del cordón (Z = 0,1 m) no podemos hacer mucho: si los sacamos quedan 2 puntos por foto, y para H hacen falta 4. Así que los tratamos como si estuvieran en el piso. La cámara está a varios metros, así que 10 cm de altura mueven el punto apenas unos píxeles en la foto.

**El problema.** Hicimos Zhang completo (`armarA` y `calcularMatrizK2D`) y la K que salió no tenía sentido: un sesgo de más de 1000 y α_v = 213. Lo verificamos con numpy y da exactamente lo mismo, así que no es un error del código: son los datos. Las 4 fotos son muy parecidas entre sí (la cámara casi no se movió) y hay solo 5 puntos por foto, así que las ecuaciones de A son casi dependientes y no alcanzan para despejar las 5 incógnitas de K.

**La simplificación.** Usamos lo mismo que en el 2d: suponemos que los píxeles son cuadrados (α_u = α_v = f), que no hay sesgo (K₁₂ = 0) y que el centro óptico está en el centro de la imagen. Así nos queda una sola incógnita, f. Si corremos los píxeles para que el centro quede en (0, 0), K = diag(f, f, 1) y B = K⁻ᵀK⁻¹ = diag(w, w, 1) con w = 1/f². Las dos condiciones de cada foto (r₁ ⟂ r₂ y ‖r₁‖ = ‖r₂‖) quedan:

- w (h₁₁h₁₂ + h₂₁h₂₂) + h₃₁h₃₂ = 0
- w (h₁₁² + h₂₁² − h₁₂² − h₂₂²) + (h₃₁² − h₃₂²) = 0

Son ecuaciones de la forma a·w + c = 0, con una sola incógnita. Con las 4 fotos tenemos 8, y las resolvemos por cuadrados mínimos: w = −Σ(a·c) / Σa².

In [ ]:
import pandas as pd
from alc import calibrarVistasSalguero, obtenerPose2D, productoMatriz, producto_matriz_vector

datos = pd.read_csv("TP_template/salguero/world_img_pts.csv", sep=";")
world_pts = datos[["X", "Y", "Z"]].values.astype(float)
vistas = []
for k in range(4):
    vistas.append(datos[[f"u_{k}", f"v_{k}"]].values.astype(float))

# El centro óptico lo ponemos en el centro de la imagen
img_salguero = mpimg.imread("TP_template/salguero/img_10_18_47_351.jpg")
alto, ancho = img_salguero.shape[0], img_salguero.shape[1]
centro = (ancho / 2, alto / 2)

K_salguero = calibrarVistasSalguero(world_pts, vistas, centro)
print("K =\n", K_salguero)

# f con cada foto por separado, para ver qué tan parejas dan
for k in range(4):
    print(f"f con la vista {k} sola: {calibrarVistasSalguero(world_pts, [vistas[k]], centro)[0, 0]:.0f} px")

# Error de reproyección: con K y la H de cada foto sacamos la pose, y proyectamos los puntos (con su Z real)
for k in range(4):
    H_k = estimarHomografiaCancha(vistas[k], world_pts[:, 0:2])
    R_k, t_k = obtenerPose2D(H_k, K_salguero)
    P_k = productoMatriz(K_salguero, np.hstack([R_k, t_k.reshape(3, 1)]))
    error = np.zeros(len(world_pts))
    for i in range(len(world_pts)):
        p = producto_matriz_vector(P_k, np.hstack([world_pts[i], 1.0]))
        error[i] = np.sqrt((p[0] / p[2] - vistas[k][i, 0])**2 + (p[1] / p[2] - vistas[k][i, 1])**2)
    print(f"Vista {k}: error de reproyección (px) = {error}")

**Resultado.** f da unos 3200 píxeles. Con cada foto por separado da parecido (entre 3060 y 3250), salvo la vista 1, que da 4076. El error de reproyección es de entre 3 y 13 píxeles. No es tan chico como en el ejercicio 2, pero tiene sentido: hay solo 5 puntos marcados a mano y estamos ignorando los 10 cm del cordón.

**Ojo: este f no nos termina de cerrar.** Lo chequeamos con los 51 puntos del 5b (otra foto de la misma esquina, todos en el piso). Con la H de esa foto y f = 3227, r₁ y r₂ no salen del mismo largo (uno mide 1,55 veces el otro), así que la R que se arma no es una rotación. Si la obligamos a ser una rotación, el error de reproyección se va a más de 100 píxeles. En cambio, haciendo la misma cuenta con esos 51 puntos, f da unos 2000 píxeles, y con f = 2000 la R sí sale una rotación y el error baja a unos 8 píxeles.

O sea que con solo 5 puntos (marcados a mano, y con 4 fotos muy parecidas) la estimación de f no es confiable. También puede ser que esas fotos se hayan sacado con otro zoom, pero con los datos que tenemos no lo podemos saber.


### 5 b) Pose de la cámara y reproyección

En `gs_10_09_point_correspondences_v5.txt` hay 51 puntos con sus (X, Y, Z) y sus (u, v) en la foto. Todos tienen Z = 0, o sea que están todos en el piso, y entonces el DLT 3D del ejercicio 2 no sirve (es el caso coplanar del 1a). Hacemos lo mismo que en el ejercicio 3: sacamos la homografía H del piso, y con la K del 5a la pose con `obtenerPose2D` (r₁ = λK⁻¹h₁, r₂ = λK⁻¹h₂, r₃ = r₁ × r₂, t = λK⁻¹h₃, con el signo que deja la escena delante de la cámara).

Como vimos en el 5a, la f de ahí no le cierra bien a esta foto: r₁ y r₂ no salen del mismo largo ni del todo perpendiculares, así que la R que se arma no es una rotación. Para arreglarlo buscamos la rotación más cercana (`proyectarARotacion`): si R = U S Vᵀ, la rotación más cercana es U Vᵀ, que es como dejar todos los valores singulares en 1.

Después proyectamos los 51 puntos con K[R | t] y los dibujamos sobre la foto, junto con los puntos marcados, para ver cuánto se alejan.

In [ ]:
from alc import calcularPoseSalguero, proyectarPuntosPinhole

datos_5b = np.loadtxt("TP_template/salguero/gs_10_09_point_correspondences_v5.txt")
world_5b = datos_5b[1:4].T   # (51, 3): X, Y, Z
img_5b = datos_5b[4:6].T     # (51, 2): u, v

R_5b, t_5b = calcularPoseSalguero(world_5b, img_5b, K_salguero)
print("R =\n", R_5b)
print("t =", t_5b)

reproy_5b = proyectarPuntosPinhole(K_salguero, R_5b, t_5b, world_5b)
error_5b = np.sqrt(np.sum((reproy_5b - img_5b)**2, axis=1))
print(f"Error de reproyección: medio {np.mean(error_5b):.1f} px, máximo {np.max(error_5b):.1f} px")

plt.figure(figsize=(12, 9))
plt.imshow(img_salguero)
plt.scatter(img_5b[:, 0], img_5b[:, 1], c="yellow", s=20, label="marcados")
plt.scatter(reproy_5b[:, 0], reproy_5b[:, 1], c="red", marker="x", s=30, label="reproyectados con K[R | t]")
plt.legend()
plt.title("Reproyección de los 51 puntos sobre img_10_18_47_351.jpg")
plt.show()

**Resultado.** La R sale una rotación, pero el error de reproyección es grande (decenas de píxeles en promedio y varios cientos en los puntos más alejados). No es un problema de la pose: es la f del 5a. Si con estos mismos 51 puntos estimamos f (da unos 2000 píxeles) y repetimos todo, el error baja a unos 8 píxeles. Así que la pose que obtenemos depende mucho de qué tan buena sea K, y con la K del 5a no alcanza para reproyectar bien esta foto.

### 5 c) Velocidad de los vehículos y de la peatona

Las capturas del video están en `TP_template/salguero/video`: son 46 fotos tomadas en unos 5 segundos (de 10:18:47,351 a 10:18:52,546). La primera es la misma foto del 5b y la cámara no se mueve, así que nos sirve la misma homografía del piso que sale de los 51 puntos.

El taxi, la camioneta y la señora se mueven sobre el piso, así que para ubicarlos alcanza con H⁻¹, que lleva un punto de la foto al piso (como en el ejercicio 4). Para eso hay que marcar en cada captura un punto que esté sobre el piso: para los vehículos, el punto medio entre donde tocan la calle las dos ruedas de adelante; para la señora, el medio entre los pies. Para que la cuenta tenga sentido, tiene que ser siempre el mismo punto del vehículo.

El tiempo de cada captura sale del nombre del archivo (`tiempoDesdeNombre`), y la velocidad entre dos capturas es la distancia recorrida sobre el piso dividida el tiempo que pasó (`velocidadPuntos`).

Un detalle a favor: para esto no hace falta K, solo H. Así que el problema que tuvimos con la f del 5a no afecta a las velocidades.

Los vehículos aparecen recién en los últimos 2 segundos, así que para ellos usamos 5 capturas separadas por unos 0,4 segundos. A la señora la seguimos durante todo el video, con una captura por segundo más o menos.

In [ ]:
from alc import tiempoDesdeNombre, velocidadPuntos

carpeta_video = "TP_template/salguero/video/"

# H del piso: la misma del 5b (la cámara del video es la misma y no se mueve)
H_piso = estimarHomografiaCancha(img_5b, world_5b[:, 0:2])

# Para cada agente: las capturas que usamos y el punto del piso marcado en cada una (u, v)
agentes = {
    "taxi": (["img_10_18_50_534.jpg", "img_10_18_50_986.jpg", "img_10_18_51_470.jpg",
              "img_10_18_51_875.jpg", "img_10_18_52_312.jpg"],
             np.array([[766, 344], [732, 365], [643, 393], [525, 439], [420, 500]])),
    "camioneta": (["img_10_18_50_534.jpg", "img_10_18_50_986.jpg", "img_10_18_51_470.jpg",
                   "img_10_18_51_875.jpg", "img_10_18_52_312.jpg"],
                  np.array([[963, 301], [947, 321], [912, 347], [894, 375], [882, 400]])),
    "señora": (["img_10_18_47_351.jpg", "img_10_18_48_350.jpg", "img_10_18_49_286.jpg",
                "img_10_18_50_206.jpg", "img_10_18_51_173.jpg", "img_10_18_52_219.jpg"],
               np.array([[420, 497], [550, 493], [645, 492], [770, 491], [862, 489], [970, 483]])),
}

colores = {"taxi": "yellow", "camioneta": "cyan", "señora": "red"}
plt.figure(figsize=(12, 9))
plt.imshow(mpimg.imread(carpeta_video + "img_10_18_52_312.jpg"))

for nombre in agentes:
    archivos, pixeles = agentes[nombre]
    tiempos = np.zeros(len(archivos))
    for i in range(len(archivos)):
        tiempos[i] = tiempoDesdeNombre(archivos[i])

    posiciones = imagenACancha(H_piso, pixeles)          # en metros, sobre el piso
    v = velocidadPuntos(posiciones, tiempos)             # m/s entre capturas seguidas

    # velocidad media: distancia entre la primera y la última posición sobre el tiempo total
    distancia = np.sqrt(np.sum((posiciones[-1] - posiciones[0])**2))
    v_media = distancia / (tiempos[-1] - tiempos[0])

    print(f"{nombre}: velocidad entre capturas (km/h) = {v * 3.6}")
    print(f"   velocidad media = {v_media:.2f} m/s = {v_media * 3.6:.1f} km/h")

    plt.plot(pixeles[:, 0], pixeles[:, 1], "o-", c=colores[nombre], label=f"{nombre}: {v_media * 3.6:.0f} km/h")

plt.legend()
plt.title("Puntos marcados en cada captura (sobre la última foto)")
plt.show()

**Resultado.** La señora camina a unos 5 km/h, que es lo normal para una persona. El taxi va a unos 48 km/h, y la camioneta a unos 61 km/h en promedio, aunque se ve que va frenando (pasa de unos 76 km/h a unos 38 km/h).

Las velocidades entre capturas seguidas saltan bastante, sobre todo la de la camioneta, y tiene sentido. Lejos de la cámara cada píxel equivale a mucho piso: donde arranca la camioneta, 1 píxel son unos 55 cm. Errarle por un par de píxeles al marcar la rueda, en un tramo de 0,4 segundos, cambia la velocidad en casi 10 km/h. Por eso la velocidad media, que usa la primera y la última posición (y por lo tanto un tramo más largo), es más confiable.

Además los primeros puntos de la camioneta están a más de 40 metros, más lejos que casi todos los 51 puntos con los que armamos H, así que ahí H está extrapolando.